# Real Estate Analysis
## Machine Learning & Prediction
* **Goal:** Develop a machine learning model for price prediction based on property characteristics and location

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    r2_score
)
from sklearn.dummy import DummyClassifier

In [2]:
# Safe Dataloading
real_estate_ml = '../data/real_estate_ml.csv'

try:
    df = pd.read_csv(real_estate_ml)
    print(f'File Loaded successfully')
except FileNotFoundError:
    print(f'File Not Found - check {real_estate_ml}')
except Exception as e:
    print(f'Error occured - check {e}')

File Loaded successfully


In [ ]:
# Check the head
df.head()

,country,location,building_construction_year_analysis,building_total_floors,apartment_floor_analysis,apartment_rooms_analysis,apartment_bedrooms_analysis,apartment_bathrooms_analysis,apartment_total_area_analysis,apartment_rooms_missing,apartment_bedrooms_missing,apartment_bathrooms_missing,building_construction_year_missing,apartment_floor_missing,building_total_floors_missing,price_in_USD
0,Turkey,"Mediterranean Region, Turkey",2022.0,5.0,1.0,3.0,2.0,2.0,120.0,0,0,0,1,0,0,315209.0
1,Turkey,"Kalkan, Mediterranean Region, Kas, Turkey",2021.0,2.0,4.0,2.0,2.0,1.0,500.0,1,1,1,0,1,0,1108667.0
2,Turkey,"Mediterranean Region, Antalya, Turkey",2022.0,5.0,2.0,2.0,1.0,1.0,65.0,0,0,0,1,0,0,173211.0
3,Thailand,"Chon Buri Province, Pattaya, Thailand",2020.0,15.0,5.0,2.0,1.0,1.0,86.0,0,0,0,0,0,0,99900.0
4,Thailand,"Chon Buri Province, Pattaya, Thailand",2026.0,8.0,3.0,3.0,2.0,1.0,86.0,0,0,0,0,0,0,67000.0


In [5]:
# Define features and target
X = df.drop('price_in_USD', axis=1)
y = df['price_in_USD']

In [6]:
# Split data into training and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# Check the shape of each set
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(115968, 15)
(28993, 15)
(115968,)
(28993,)


In [8]:
# Check location before encoding
print(X_train['location'].nunique())
print(X_train['location'].value_counts().head(10))

6546
location
Mediterranean Region, Sekerhane Mahallesi, Alanya, Turkey                                                5797
Central Federal District, poselenie Sosenskoe, Novomoskovsky Administrative Okrug, Russia                5583
Central Hungary, Budapest, Komarom-Esztergom, Hungary                                                    3514
Minsk, Belarus                                                                                           3341
Kommunarka, Central Federal District, poselenie Sosenskoe, Novomoskovsky Administrative Okrug, Russia    2171
Transdanubia, Budapest, Komarom-Esztergom, Hungary                                                       2030
Dubai, UAE                                                                                               1963
Montenegro                                                                                               1733
Brest Region, Brest, Belarus                                                                             1

### Encoding categorical features

Machine learning models cannot directly work with text-based categorical features.

For `country`, One-Hot Encoding is suitable because there are only 27 different categories.

For `location`, there are more than 6,500 categories. One-Hot Encoding would create thousands of additional columns. Therefore, frequency encoding is used to represent each location by how frequently it occurs in the training data.

In [9]:
# Frequency encoding for location

location_frequency = X_train['location'].value_counts()

X_train['location_frequency'] =  X_train['location'].map(location_frequency)
X_test['location_frequency'] =  X_test['location'].map(location_frequency)

X_train['location_frequency'] =  X_train['location_frequency'].fillna(0)
X_test['location_frequency'] =  X_test['location_frequency'].fillna(0)

In [10]:
# One Hot Encoding for country

X_train = pd.get_dummies(
    X_train, 
    columns=['country'], 
    dtype=int
)

X_test = pd.get_dummies(
    X_test, 
    columns=['country'], 
    dtype=int
)

In [11]:
# Align X_test and X_train country column

X_train, X_test = X_train.align(
    X_test, 
    join='left', 
    axis=1, 
    fill_value=0
)

In [12]:
# Check shape 

print(X_train.shape)
print(X_test.shape)
print(X_train.dtypes.value_counts())

(115968, 43)
(28993, 43)
int64      35
float64     7
object      1
Name: count, dtype: int64


In [13]:
# Remove original location column

X_train = X_train.drop('location', axis=1)
X_test = X_test.drop('location', axis=1)

In [14]:
print(X_train.shape)
print(X_test.shape)
print(X_train.dtypes.value_counts())

(115968, 42)
(28993, 42)
int64      35
float64     7
Name: count, dtype: int64


In [15]:
# Start with Baseline Model
from sklearn.dummy import DummyRegressor

In [16]:
# Create Baseline Model

baseline = DummyRegressor(strategy='mean')

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)